# Notebook 2: RAG — Knowledge Retrieval with Pinecone

**Technique covered:** Transformer-based Models / LLMs (embeddings + RAG)

This notebook demonstrates the Retrieval-Augmented Generation (RAG) pipeline:
1. Load domain knowledge from JSON into a Pinecone vector index
2. Encode knowledge chunks with OpenAI `text-embedding-3-large` (3072-dim)
3. Perform semantic search to retrieve relevant context
4. Show how retrieved context improves question generation quality

In [1]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from openai import OpenAI
import numpy as np
from pathlib import Path

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
EMBED_MODEL = 'text-embedding-3-large'

print('OpenAI client ready')

OpenAI client ready


## 1. Load Domain Knowledge

In [2]:
DATA_DIR = Path('../data')

def load_domain(domain: str) -> list[dict]:
    path = DATA_DIR / f'{domain}.json'
    with open(path) as f:
        data = json.load(f)
    return data['topics']

se_topics = load_domain('software_engineering')
print(f'Loaded {len(se_topics)} topics for software_engineering')
print('\nFirst topic:')
print(f'  Concept    : {se_topics[0]["concept"]}')
print(f'  Difficulty : {se_topics[0]["difficulty"]}')
print(f'  Explanation: {se_topics[0]["explanation"][:100]}...')

Loaded 15 topics for software_engineering

First topic:
  Concept    : Data Structures â€” Arrays vs Linked Lists
  Difficulty : easy
  Explanation: Arrays store elements in contiguous memory, giving O(1) random access but O(n) insertion/deletion. L...


## 2. Generate Embeddings

In [3]:
def embed(text: str) -> np.ndarray:
    result = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(result.data[0].embedding)

# Embed all topics
knowledge_store = []
for topic in se_topics:
    text = f"{topic['concept']}: {topic['explanation']}"
    knowledge_store.append({
        **topic,
        'embedding': embed(text),
        'full_text': text
    })

print(f'Embedded {len(knowledge_store)} knowledge chunks')
print(f'Embedding dimension: {len(knowledge_store[0]["embedding"])}')

Embedded 15 knowledge chunks
Embedding dimension: 3072


## 3. Semantic Retrieval (In-Memory Demo)

In [4]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query: str, store: list, top_k: int = 3) -> list[dict]:
    q_emb = embed(query)
    scored = [
        {**item, 'score': cosine_similarity(q_emb, item['embedding'])}
        for item in store
    ]
    return sorted(scored, key=lambda x: x['score'], reverse=True)[:top_k]

# Test retrieval
query = "How do distributed systems handle inconsistent data?"
results = retrieve(query, knowledge_store, top_k=3)

print(f'Query: "{query}"\n')
for i, r in enumerate(results, 1):
    print(f'Rank {i} (score={r["score"]:.4f})')
    print(f'  Concept    : {r["concept"]}')
    print(f'  Difficulty : {r["difficulty"]}')
    print(f'  Explanation: {r["explanation"][:120]}...')
    print()

Query: "How do distributed systems handle inconsistent data?"

Rank 1 (score=0.5072)
  Concept    : Distributed Systems â€” CAP Theorem
  Difficulty : hard
  Explanation: A distributed system can guarantee at most 2 of 3 properties: Consistency (all nodes see same data), Availability (every...

Rank 2 (score=0.3540)
  Concept    : Caching Strategies
  Difficulty : medium
  Explanation: Cache-aside (lazy loading): app checks cache, falls back to DB. Write-through: writes go to cache and DB simultaneously....

Rank 3 (score=0.3343)
  Concept    : Concurrency â€” Deadlock, Race Conditions, Mutex
  Difficulty : hard
  Explanation: Race condition: two threads access shared data concurrently with at least one write. Mutex (mutual exclusion lock) seria...



## 4. RAG vs No-RAG: Question Generation Quality

In [5]:

def generate_question(context: str, difficulty: str) -> str:
    prompt = f"""Generate one {difficulty}-level technical interview question for a software engineering role.

Context from domain knowledge:
{context}

Output ONLY the question text."""
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0.7, max_tokens=150)
    return response.choices[0].message.content.strip()

# Without RAG
no_rag_q = generate_question('No specific context.', 'hard')

# With RAG
rag_context = '\n'.join(f"- {r['concept']}: {r['explanation'][:150]}" for r in results)
rag_q = generate_question(rag_context, 'hard')

print('WITHOUT RAG:')
print(f'  {no_rag_q}')
print()
print('WITH RAG (context from distributed systems retrieval):')
print(f'  {rag_q}')

WITHOUT RAG:
  Explain how you would design a distributed system to handle high availability and fault tolerance. What specific strategies would you use to ensure data consistency and partition tolerance in the presence of network failures?

WITH RAG (context from distributed systems retrieval):
  Explain how you would design a distributed caching system that adheres to the CAP theorem, ensuring both availability and partition tolerance while managing the challenges of consistency. What strategies would you implement to handle race conditions and prevent deadlocks in this system?


## 5. Pinecone Integration (Production Path)

In [6]:
# This cell shows the production Pinecone ingestion — requires PINECONE_API_KEY in .env
import os

PINECONE_KEY = os.getenv('PINECONE_API_KEY')

if PINECONE_KEY:
    from rag.knowledge_base import KnowledgeBase
    kb = KnowledgeBase()
    n = kb.load_domain('software_engineering')
    print(f'Upserted {n} vectors into Pinecone')
    
    # Live query
    live_results = kb.query("concurrency and race conditions", domain='software_engineering', top_k=2)
    for r in live_results:
        print(f"  {r['concept']} (score={r['relevance_score']})")
else:
    print('PINECONE_API_KEY not set — skipping live Pinecone demo.')
    print('Set it in backend/.env to run the production pipeline.')

Upserted 15 vectors into Pinecone


  Concurrency — Deadlock, Race Conditions, Mutex (score=0.6548)
  Distributed Systems — CAP Theorem (score=0.3269)


## 6. Retrieval Quality Analysis

In [7]:
import pandas as pd

test_queries = [
    ("caching and performance", ["Caching Strategies"]),
    ("database transactions ACID", ["SQL vs NoSQL Databases"]),
    ("class design principles", ["SOLID Principles", "Design Patterns"]),
    ("distributed consistency tradeoffs", ["Distributed Systems — CAP Theorem", "Microservices Architecture"]),
]

rows = []
for query, expected in test_queries:
    top1 = retrieve(query, knowledge_store, top_k=1)[0]
    hit = any(e in top1['concept'] for e in expected)
    rows.append({'Query': query[:45], 'Top-1 Retrieved': top1['concept'][:40], 'Score': round(top1['score'], 4), 'Hit': '✓' if hit else '✗'})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nPrecision@1: {df['Hit'].str.count('✓').sum() / len(df):.0%}")

                            Query                     Top-1 Retrieved  Score Hit
          caching and performance                  Caching Strategies 0.5296   ✓
       database transactions ACID              SQL vs NoSQL Databases 0.4021   ✓
          class design principles                    SOLID Principles 0.5590   ✓
distributed consistency tradeoffs Distributed Systems â€” CAP Theorem 0.5300   ✗

Precision@1: 75%


## Summary

The RAG pipeline:
- Embeds 15+ domain knowledge chunks using OpenAI transformer embeddings (3072 dimensions)
- Retrieves semantically relevant context in real-time for each user turn
- Grounds question generation in accurate domain knowledge (reducing hallucination)
- Achieves high Precision@1 on diverse query types

In production, Pinecone handles the vector store with millisecond query latency.